# 综合练习：设备运行风险综合分析

## 一、练习背景

现在有两张表：

```text
df_device：设备台账表
df_log：设备运行日志表
```

`df_device` 记录设备的基础信息，包括设备所属站点、跑道方向、设备类型、型号和投产日期。

`df_log` 记录设备每天的运行状态和报警次数。

现在需要基于这两张表，完成一次设备级别的综合分析，识别：

```text
1. 哪些设备累计报警较多；
2. 哪些设备出现过 ERROR；
3. 哪些设备存在连续 ERROR；
4. 哪些设备最近状态异常；
5. 哪些日志设备没有登记在设备台账中；
6. 哪些登记设备没有任何日志；
7. 哪些设备整体运行风险较高。
```

本题不是多个小题，而是要求你最终生成一张完整的设备级综合分析结果表。

---

## 二、输入表

### 1. 设备台账表：df_device

字段如下：

```text
device_id
site
runway
device_type
model
install_date
```

字段含义：

| 字段 | 含义 |
|---|---|
| device_id | 设备编号 |
| site | 所属站点 |
| runway | 所属跑道方向 |
| device_type | 设备类型 |
| model | 设备型号 |
| install_date | 投产日期 |

---

### 2. 设备运行日志表：df_log

字段如下：

```text
device_id
stat_date
status
alarm_count
```

字段含义：

| 字段 | 含义 |
|---|---|
| device_id | 设备编号 |
| stat_date | 日志日期 |
| status | 当日运行状态 |
| alarm_count | 当日报警次数 |

`status` 可能取值：

```text
NORMAL
WARN
ERROR
```

---

## 三、最终分析目标

请生成一张设备级别的综合分析结果表。

每一行代表一台设备。

最终结果表需要覆盖：

```text
1. 设备台账中登记过的设备；
2. 日志表中出现过但台账中没有登记的设备。
```

也就是说，最终设备集合应该来自：

```text
df_device.device_id
+
df_log.device_id
```

而不是只看其中一张表。

---

## 四、最终输出字段

最终结果表字段要求如下：

```text
device_id
site
runway
device_type
model
has_device_info
has_log
first_log_date
latest_stat_date
latest_status
total_alarm_count
error_days
warn_days
max_consecutive_error_days
equipment_risk_score
risk_rank
risk_level
data_quality_status
```

---

## 五、字段计算规则

### 1. has_device_info

表示该设备是否存在于设备台账表 `df_device` 中。

规则：

```text
如果 device_id 能在 df_device 中找到，has_device_info = True
否则 has_device_info = False
```

例如：

```text
日志表中出现了某个设备，但台账表没有这个设备，
则 has_device_info = False。
```

---

### 2. has_log

表示该设备是否存在运行日志。

规则：

```text
如果 device_id 能在 df_log 中找到，has_log = True
否则 has_log = False
```

例如：

```text
设备台账中登记了某个设备，
但日志表中完全没有这个设备的记录，
则 has_log = False。
```

---

### 3. first_log_date

该设备最早一条日志日期。

规则：

```text
按 device_id 分组，取 stat_date 的最小值。
```

如果设备没有日志，则为空。

---

### 4. latest_stat_date

该设备最近一条日志日期。

规则：

```text
按 device_id 分组，取 stat_date 的最大值。
```

如果设备没有日志，则为空。

---

### 5. latest_status

该设备最近一条日志对应的状态。

规则：

```text
按 device_id 分组；
按照 stat_date 降序排序；
取每个设备最近一条日志的 status。
```

如果设备没有日志，则为空。

---

### 6. total_alarm_count

设备累计报警次数。

规则：

```text
按 device_id 分组；
对 alarm_count 求和。
```

如果设备没有日志，则记为 0。

---

### 7. error_days

设备出现 ERROR 的天数。

规则：

```text
统计 status = 'ERROR' 的记录数。
```

如果设备没有日志，则记为 0。

---

### 8. warn_days

设备出现 WARN 的天数。

规则：

```text
统计 status = 'WARN' 的记录数。
```

如果设备没有日志，则记为 0。

---

### 9. max_consecutive_error_days

设备最长连续 ERROR 天数。

规则：

```text
只统计 status = 'ERROR' 的日期。

连续 ERROR 必须按自然日期连续。
如果两个 ERROR 之间日期断档，则不能算作同一个连续段。
```

例如：

| device_id | stat_date | status |
|---|---|---|
| A | 2026-07-01 | ERROR |
| A | 2026-07-02 | ERROR |
| A | 2026-07-04 | ERROR |

其中：

```text
2026-07-01 到 2026-07-02 是连续 ERROR，长度为 2。
2026-07-04 虽然也是 ERROR，但中间缺少 2026-07-03，不能接上前面的连续段。
```

所以该设备最长连续 ERROR 天数为：

```text
2
```

如果设备从未出现 ERROR，则记为 0。

---

### 10. equipment_risk_score

设备运行风险分数。

计算公式：

```text
equipment_risk_score
=
total_alarm_count
+ error_days * 5
+ max_consecutive_error_days * 10
```

说明：

```text
累计报警次数越多，风险越高；
ERROR 天数越多，风险越高；
连续 ERROR 越长，风险越高。
```

如果设备没有日志，则风险分数记为 0。

---

### 11. risk_rank

设备风险排名。

规则：

```text
按照 equipment_risk_score 从高到低排名。
如果风险分数相同，使用并列排名。
```

也就是说：

```text
SQL 轨道使用 RANK()
Pandas 轨道使用 rank(method='min', ascending=False)
```

---

### 12. risk_level

设备风险等级。

规则如下：

| 条件 | risk_level |
|---|---|
| has_log = False | NO_LOG |
| latest_status = 'ERROR' | HIGH |
| max_consecutive_error_days >= 2 | HIGH |
| equipment_risk_score >= 35 | HIGH |
| error_days >= 1 | MEDIUM |
| latest_status = 'WARN' | MEDIUM |
| total_alarm_count >= 10 | MEDIUM |
| 其他情况 | LOW |

判断顺序从上到下。

---

### 13. data_quality_status

数据质量状态。

规则如下：

| 条件 | data_quality_status |
|---|---|
| has_device_info = False | UNKNOWN_DEVICE |
| has_log = False | NO_LOG |
| 其他情况 | OK |

说明：

```text
UNKNOWN_DEVICE：
日志表中出现了设备，但设备台账中没有登记。

NO_LOG：
设备台账中登记了设备，但日志表中没有任何日志。

OK：
设备台账和日志均能正常匹配。
```

---

## 六、业务要求

### 1. 最终结果必须覆盖完整设备集合

最终结果不能只来自 `df_device`，也不能只来自 `df_log`。

必须同时覆盖：

```text
df_device 中登记过的设备
df_log 中出现过的设备
```

---

### 2. 不能使用 INNER JOIN 作为最终设备集合

原因：

```text
INNER JOIN 会丢掉两类重要数据：

1. 日志表中有，但设备台账中没有的设备；
2. 设备台账中有，但日志表中没有的设备。
```

这两类数据本身就是数据质量问题，不能直接删除。

---

### 3. 连续 ERROR 必须考虑日期是否真正连续

不能简单使用相邻记录判断连续。

必须判断：

```text
当前 ERROR 日期
是否等于上一条 ERROR 日期 + 1 天
```

如果日期断档，则要开启新的连续段。

---

### 4. 最近状态必须取最近日期对应的 status

不能直接对 status 做 max 或 min。

必须先找到每个设备最近一条日志，再取该日志对应的状态。

---

### 5. 最终结果排序

最终结果按以下顺序排序：

```text
risk_rank 升序
device_id 升序
```

也就是风险排名越靠前的设备越先显示。

---

## 七、本题综合训练点

本题综合使用以下知识：

```text
1. Join / Full coverage 思维
2. Union 设备全集
3. Group By 聚合
4. 条件计数
5. 每组最新记录
6. Gap & Island 连续区间识别
7. Time Comparison 日期连续性判断
8. Ranking 风险排名
9. EXISTS / NOT EXISTS 思维
10. 数据质量标记
11. 字段构造
12. SQL 与 Pandas 双轨实现
```

---

## 八、完成要求

请分别完成：

```text
SQL 轨道
Pandas 轨道
```

最终分别输出：

```text
df_sql
df_pd
```

两个结果在业务含义上应保持一致。

In [1]:
import pandas as pd
import duckdb

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# =========================
# 设备台账表：df_device
# =========================

df_device = pd.DataFrame(
    [
        {
            "device_id": "VIS_A",
            "site": "R34",
            "runway": "RWY34",
            "device_type": "VIS",
            "model": "FS11",
            "install_date": "2023-03-12",
        },
        {
            "device_id": "RVR_B",
            "site": "R34",
            "runway": "RWY34",
            "device_type": "RVR",
            "model": "LT31",
            "install_date": "2022-11-08",
        },
        {
            "device_id": "VIS_C",
            "site": "R35",
            "runway": "RWY35",
            "device_type": "VIS",
            "model": "FS11",
            "install_date": "2021-06-20",
        },
        {
            "device_id": "RVR_D",
            "site": "R35",
            "runway": "RWY35",
            "device_type": "RVR",
            "model": "LT31",
            "install_date": "2024-01-15",
        },
        {
            "device_id": "BLS_E",
            "site": "R34",
            "runway": "RWY34",
            "device_type": "BLS",
            "model": "BL100",
            "install_date": "2020-09-01",
        },
        {
            "device_id": "VIS_F",
            "site": "R36",
            "runway": "RWY36",
            "device_type": "VIS",
            "model": "FS11",
            "install_date": "2025-02-10",
        },
    ]
)

df_device["install_date"] = pd.to_datetime(df_device["install_date"])


# =========================
# 设备运行日志表：df_log
# =========================

df_log = pd.DataFrame(
    [
        # VIS_A：有多次 ERROR，并且 7月5日-7月7日形成连续 ERROR 3 天
        {"device_id": "VIS_A", "stat_date": "2026-07-01", "status": "NORMAL", "alarm_count": 1},
        {"device_id": "VIS_A", "stat_date": "2026-07-02", "status": "ERROR",  "alarm_count": 8},
        {"device_id": "VIS_A", "stat_date": "2026-07-03", "status": "ERROR",  "alarm_count": 7},
        {"device_id": "VIS_A", "stat_date": "2026-07-04", "status": "NORMAL", "alarm_count": 2},
        {"device_id": "VIS_A", "stat_date": "2026-07-05", "status": "ERROR",  "alarm_count": 5},
        {"device_id": "VIS_A", "stat_date": "2026-07-06", "status": "ERROR",  "alarm_count": 6},
        {"device_id": "VIS_A", "stat_date": "2026-07-07", "status": "ERROR",  "alarm_count": 4},

        # RVR_B：有日志，但从未 ERROR；中间存在日期缺口
        {"device_id": "RVR_B", "stat_date": "2026-07-01", "status": "NORMAL", "alarm_count": 0},
        {"device_id": "RVR_B", "stat_date": "2026-07-02", "status": "NORMAL", "alarm_count": 1},
        {"device_id": "RVR_B", "stat_date": "2026-07-04", "status": "WARN",   "alarm_count": 3},
        {"device_id": "RVR_B", "stat_date": "2026-07-05", "status": "NORMAL", "alarm_count": 0},
        {"device_id": "RVR_B", "stat_date": "2026-07-07", "status": "NORMAL", "alarm_count": 1},

        # VIS_C：出现 ERROR，但 7月1日 和 7月4日之间日期断档，不能算连续
        {"device_id": "VIS_C", "stat_date": "2026-07-01", "status": "ERROR",  "alarm_count": 10},
        {"device_id": "VIS_C", "stat_date": "2026-07-02", "status": "NORMAL", "alarm_count": 3},
        {"device_id": "VIS_C", "stat_date": "2026-07-04", "status": "ERROR",  "alarm_count": 8},
        {"device_id": "VIS_C", "stat_date": "2026-07-05", "status": "ERROR",  "alarm_count": 9},
        {"device_id": "VIS_C", "stat_date": "2026-07-07", "status": "NORMAL", "alarm_count": 2},
        {"device_id": "VIS_C", "stat_date": "2026-07-08", "status": "ERROR",  "alarm_count": 8},
        {"device_id": "VIS_C", "stat_date": "2026-07-09", "status": "ERROR",  "alarm_count": 8},
        {"device_id": "VIS_C", "stat_date": "2026-07-10", "status": "ERROR",  "alarm_count": 8},
        {"device_id": "VIS_C", "stat_date": "2026-07-12", "status": "ERROR",  "alarm_count": 8},
        {"device_id": "VIS_C", "stat_date": "2026-07-15", "status": "ERROR",  "alarm_count": 8},
        {"device_id": "VIS_C", "stat_date": "2026-07-16", "status": "ERROR",  "alarm_count": 8},

        # BLS_E：有 WARN，也有单日 ERROR
        {"device_id": "BLS_E", "stat_date": "2026-07-01", "status": "NORMAL", "alarm_count": 0},
        {"device_id": "BLS_E", "stat_date": "2026-07-02", "status": "WARN",   "alarm_count": 2},
        {"device_id": "BLS_E", "stat_date": "2026-07-03", "status": "ERROR",  "alarm_count": 4},
        {"device_id": "BLS_E", "stat_date": "2026-07-06", "status": "NORMAL", "alarm_count": 1},

        # VIS_F：没有 ERROR，但最近状态是 WARN
        {"device_id": "VIS_F", "stat_date": "2026-07-03", "status": "NORMAL", "alarm_count": 0},
        {"device_id": "VIS_F", "stat_date": "2026-07-04", "status": "NORMAL", "alarm_count": 1},
        {"device_id": "VIS_F", "stat_date": "2026-07-05", "status": "WARN",   "alarm_count": 2},

        # UNK_X：日志表中存在，但设备台账中没有登记；并且连续 ERROR 2 天
        {"device_id": "UNK_X", "stat_date": "2026-07-01", "status": "ERROR",  "alarm_count": 6},
        {"device_id": "UNK_X", "stat_date": "2026-07-02", "status": "ERROR",  "alarm_count": 5},
        {"device_id": "UNK_X", "stat_date": "2026-07-03", "status": "NORMAL", "alarm_count": 1},

        # UNK_Y：日志表中存在，但设备台账中没有登记；没有 ERROR
        {"device_id": "UNK_Y", "stat_date": "2026-07-04", "status": "NORMAL", "alarm_count": 0},
    ]
)

df_log["stat_date"] = pd.to_datetime(df_log["stat_date"])


# =========================
# 查看数据
# =========================

df_device, df_log

(  device_id site runway device_type  model install_date
 0     VIS_A  R34  RWY34         VIS   FS11   2023-03-12
 1     RVR_B  R34  RWY34         RVR   LT31   2022-11-08
 2     VIS_C  R35  RWY35         VIS   FS11   2021-06-20
 3     RVR_D  R35  RWY35         RVR   LT31   2024-01-15
 4     BLS_E  R34  RWY34         BLS  BL100   2020-09-01
 5     VIS_F  R36  RWY36         VIS   FS11   2025-02-10,
    device_id  stat_date  status  alarm_count
 0      VIS_A 2026-07-01  NORMAL            1
 1      VIS_A 2026-07-02   ERROR            8
 2      VIS_A 2026-07-03   ERROR            7
 3      VIS_A 2026-07-04  NORMAL            2
 4      VIS_A 2026-07-05   ERROR            5
 5      VIS_A 2026-07-06   ERROR            6
 6      VIS_A 2026-07-07   ERROR            4
 7      RVR_B 2026-07-01  NORMAL            0
 8      RVR_B 2026-07-02  NORMAL            1
 9      RVR_B 2026-07-04    WARN            3
 10     RVR_B 2026-07-05  NORMAL            0
 11     RVR_B 2026-07-07  NORMAL            1
 1

In [2]:
# =================
# SQL轨道（第一步）
# =================

# 创建device_id是否分别存在df_device,df_log的标记表

query_device_flag = '''

WITH all_devices AS(
    SELECT device_id
    FROM df_device

    UNION
    
    SELECT device_id
    FROM df_log
),
device_has_id AS(
SELECT 
    ad.device_id,
    CASE 
        WHEN EXISTS(
            SELECT
                1
            FROM df_device AS dv
            WHERE dv.device_id = ad.device_id
        )
        THEN True
        ELSE False
    END  AS has_device_info

FROM all_devices AS ad
)
SELECT
    dhi.device_id,
    dhi.has_device_info,
    CASE
        WHEN EXISTS(
            SELECT
                1
            FROM df_log AS lo
            WHERE lo.device_id = dhi.device_id
        )
        THEN True
        ELSE False
    END AS has_log
FROM device_has_id AS dhi
ORDER BY device_id
'''
df_device_flag = duckdb.execute(query_device_flag).fetchdf()
df_device_flag

,device_id,has_device_info,has_log
0,BLS_E,True,True
1,RVR_B,True,True
2,RVR_D,True,False
3,UNK_X,False,True
4,UNK_Y,False,True
5,VIS_A,True,True
6,VIS_C,True,True
7,VIS_F,True,True


In [3]:
# =================
# SQL轨道（第二步）
# =================

# 创建基础信息表

query_device_base = """

SELECT
    f.device_id,
    dv.site,
    dv.runway,
    dv.device_type,
    dv.model,
    f.has_device_info,
    f.has_log
FROM df_device_flag AS f
LEFT JOIN df_device AS dv
    ON f.device_id = dv.device_id
ORDER BY f.device_id;
"""
df_device_base = duckdb.execute(query_device_base).fetchdf()
df_device_base

,device_id,site,runway,device_type,model,has_device_info,has_log
0,BLS_E,R34,RWY34,BLS,BL100,True,True
1,RVR_B,R34,RWY34,RVR,LT31,True,True
2,RVR_D,R35,RWY35,RVR,LT31,True,False
3,UNK_X,NaN,NaN,NaN,NaN,False,True
4,UNK_Y,NaN,NaN,NaN,NaN,False,True
5,VIS_A,R34,RWY34,VIS,FS11,True,True
6,VIS_C,R35,RWY35,VIS,FS11,True,True
7,VIS_F,R36,RWY36,VIS,FS11,True,True


In [4]:
# ========================================
# SQL轨道（第三步：连续日期下的最大连续ERROR）
# ========================================

query_consecutive_error = '''

WITH is_error_table AS (

    SELECT
        device_id,
        stat_date,
        status,
        alarm_count,
        CASE
            WHEN status = 'ERROR'
            THEN True
            ELSE False
        END AS is_error
    FROM df_log
),
previous_table AS(
    SELECT
        device_id,
        stat_date,
        status,
        alarm_count,
        is_error,
        COALESCE(LAG(is_error)
        OVER(PARTITION BY device_id ORDER BY stat_date),False) AS previous_is_error,
        LAG(stat_date)
        OVER(PARTITION BY device_id ORDER BY stat_date) AS previous_date
    FROM is_error_table
),
error_start_table AS(
    SELECT
        device_id,
        stat_date,
        status,
        alarm_count,
        is_error,
        previous_is_error,
        previous_date,
        CASE
            WHEN (is_error = True AND previous_is_error = False)
             OR stat_date > previous_date + INTERVAL 1 DAY
            THEN 1
            ELSE 0
        END AS error_start_sign
    FROM previous_table
    WHERE is_error = True
),
phase_table AS(
    SELECT 
        device_id,
        stat_date,
        status,
        alarm_count,
        is_error,
        previous_is_error,
        previous_date,
        error_start_sign,
        SUM(error_start_sign)
        OVER(
            PARTITION BY device_id ORDER BY stat_date
        )::INTEGER AS phase_sign
    FROM error_start_table
),
error_phase_count AS(
    SELECT
        device_id,
        COUNT(*) AS error_consecutive
    FROM phase_table
    GROUP BY device_id,phase_sign
)
SELECT 
    device_id,
    MAX(error_consecutive) AS max_consecutive_error_days
FROM error_phase_count
GROUP BY device_id

'''
df_consecutive_error = duckdb.execute(query_consecutive_error).fetchdf()
df_consecutive_error

,device_id,max_consecutive_error_days
0,VIS_A,3
1,UNK_X,2
2,VIS_C,3
3,BLS_E,1


In [5]:
# ==========================================
# SQL轨道（第四步：equipment_risk_score）
# ==========================================

query_count = '''

WITH total_alarm AS (
    SELECT
        device_id,
        SUM(alarm_count)::INTEGER AS total_alarm_count,
        SUM(
        CASE
            WHEN status = 'ERROR'
            THEN 1
            ELSE 0
        END 
        )::INTEGER AS error_days
    FROM df_log
    GROUP BY device_id
),
merge_table AS(

    SELECT
        ta.device_id,
        ta.total_alarm_count,
        ta.error_days,
        COALESCE(dce.max_consecutive_error_days, 0)::INTEGER AS max_consecutive_error_days
    FROM total_alarm AS ta
    LEFT JOIN df_consecutive_error AS dce
    ON ta.device_id = dce.device_id
)
SELECT
    device_id,
    total_alarm_count,
    error_days,
    max_consecutive_error_days,
    total_alarm_count + error_days * 5 + max_consecutive_error_days * 10 AS equipment_risk_score
FROM merge_table
ORDER BY device_id
'''
df_count = duckdb.execute(query_count).fetchdf()
df_count

,device_id,total_alarm_count,error_days,max_consecutive_error_days,equipment_risk_score
0,BLS_E,7,1,1,22
1,RVR_B,5,0,0,5
2,UNK_X,12,2,2,42
3,UNK_Y,0,0,0,0
4,VIS_A,33,5,3,88
5,VIS_C,80,9,3,155
6,VIS_F,3,0,0,3


In [6]:
# ==========================================
# SQL轨道（第五步：风险排名）
# ==========================================

query_risk_rank = '''

SELECT 
    *,
    RANK()
    OVER(ORDER BY equipment_risk_score DESC) AS risk_rank


FROM df_count
'''
df_risk_rank = duckdb.execute(query_risk_rank).fetchdf()
df_risk_rank

,device_id,total_alarm_count,error_days,max_consecutive_error_days,equipment_risk_score,risk_rank
0,VIS_C,80,9,3,155,1
1,VIS_A,33,5,3,88,2
2,UNK_X,12,2,2,42,3
3,BLS_E,7,1,1,22,4
4,RVR_B,5,0,0,5,5
5,VIS_F,3,0,0,3,6
6,UNK_Y,0,0,0,0,7


In [21]:
# ==========================================
# SQL轨道（补充：df_log_summary）
# ==========================================

query_log_summary = '''
SELECT
    device_id,
    MIN(stat_date) AS first_log_date,
    MAX(stat_date) AS latest_stat_date,
    SUM(alarm_count)::INTEGER AS total_alarm_count,
    SUM(
        CASE
            WHEN status = 'ERROR'
            THEN 1
            ELSE 0
        END
    )::INTEGER AS error_days,
    SUM(
        CASE
            WHEN status = 'WARN'
            THEN 1
            ELSE 0
        END
    )::INTEGER AS warn_days
FROM df_log
GROUP BY device_id
ORDER BY device_id
'''

df_log_summary = duckdb.execute(query_log_summary).fetchdf()
df_log_summary

,device_id,first_log_date,latest_stat_date,total_alarm_count,error_days,warn_days
0,BLS_E,2026-07-01,2026-07-06,7,1,1
1,RVR_B,2026-07-01,2026-07-07,5,0,1
2,UNK_X,2026-07-01,2026-07-03,12,2,0
3,UNK_Y,2026-07-04,2026-07-04,0,0,0
4,VIS_A,2026-07-01,2026-07-07,33,5,0
5,VIS_C,2026-07-01,2026-07-16,80,9,0
6,VIS_F,2026-07-03,2026-07-05,3,0,1


In [17]:
# ==========================================
# SQL轨道（补充：latest_status）
# ==========================================

query_latest_status = '''
WITH ranked_log AS (
    SELECT
        device_id,
        stat_date,
        status,
        ROW_NUMBER() OVER (
            PARTITION BY device_id
            ORDER BY stat_date DESC
        ) AS rn
    FROM df_log
)

SELECT
    device_id,
    stat_date AS latest_stat_date,
    status AS latest_status
FROM ranked_log
WHERE rn = 1
ORDER BY device_id
'''

df_latest_status = duckdb.execute(query_latest_status).fetchdf()
df_latest_status

,device_id,latest_stat_date,latest_status
0,BLS_E,2026-07-06,NORMAL
1,RVR_B,2026-07-07,NORMAL
2,UNK_X,2026-07-03,NORMAL
3,UNK_Y,2026-07-04,NORMAL
4,VIS_A,2026-07-07,ERROR
5,VIS_C,2026-07-16,ERROR
6,VIS_F,2026-07-05,WARN


In [22]:
# ==========================================
# SQL轨道（第六步：合并所有表）
# ==========================================

query_final_merge = '''
WITH final_base AS (
    SELECT
        ddb.device_id,
        ddb.site,
        ddb.runway,
        ddb.device_type,
        ddb.model,
        ddb.has_device_info,
        ddb.has_log,

        log.first_log_date,
        log.latest_stat_date,
        ls.latest_status,

        COALESCE(log.total_alarm_count, 0)::INTEGER AS total_alarm_count,
        COALESCE(log.error_days, 0)::INTEGER AS error_days,
        COALESCE(log.warn_days, 0)::INTEGER AS warn_days,
        COALESCE(dce.max_consecutive_error_days, 0)::INTEGER AS max_consecutive_error_days,

        (
            COALESCE(log.total_alarm_count, 0)
            + COALESCE(log.error_days, 0) * 5
            + COALESCE(dce.max_consecutive_error_days, 0) * 10
        )::INTEGER AS equipment_risk_score

    FROM df_device_base AS ddb

    LEFT JOIN df_log_summary AS log
        ON ddb.device_id = log.device_id

    LEFT JOIN df_latest_status AS ls
        ON ddb.device_id = ls.device_id

    LEFT JOIN df_consecutive_error AS dce
        ON ddb.device_id = dce.device_id
),

rank_table AS (
    SELECT
        *,
        RANK() OVER (
            ORDER BY equipment_risk_score DESC
        ) AS risk_rank
    FROM final_base
)

SELECT
    device_id,
    site,
    runway,
    device_type,
    model,
    has_device_info,
    has_log,
    first_log_date,
    latest_stat_date,
    latest_status,
    total_alarm_count,
    error_days,
    warn_days,
    max_consecutive_error_days,
    equipment_risk_score,
    risk_rank
FROM rank_table
ORDER BY risk_rank, device_id
'''

df_final_merge = duckdb.execute(query_final_merge).fetchdf()
df_final_merge

,device_id,site,runway,device_type,model,has_device_info,has_log,first_log_date,latest_stat_date,latest_status,total_alarm_count,error_days,warn_days,max_consecutive_error_days,equipment_risk_score,risk_rank
0,VIS_C,R35,RWY35,VIS,FS11,True,True,2026-07-01,2026-07-16,ERROR,80,9,0,3,155,1
1,VIS_A,R34,RWY34,VIS,FS11,True,True,2026-07-01,2026-07-07,ERROR,33,5,0,3,88,2
2,UNK_X,NaN,NaN,NaN,NaN,False,True,2026-07-01,2026-07-03,NORMAL,12,2,0,2,42,3
3,BLS_E,R34,RWY34,BLS,BL100,True,True,2026-07-01,2026-07-06,NORMAL,7,1,1,1,22,4
4,RVR_B,R34,RWY34,RVR,LT31,True,True,2026-07-01,2026-07-07,NORMAL,5,0,1,0,5,5
5,VIS_F,R36,RWY36,VIS,FS11,True,True,2026-07-03,2026-07-05,WARN,3,0,1,0,3,6
6,RVR_D,R35,RWY35,RVR,LT31,True,False,NaT,NaT,NaN,0,0,0,0,0,7
7,UNK_Y,NaN,NaN,NaN,NaN,False,True,2026-07-04,2026-07-04,NORMAL,0,0,0,0,0,7


In [24]:
# ==========================================
# SQL轨道（第七步：风险等级）
# ==========================================
query_risk_level = '''

SELECT
    device_id,
    site,
    runway,
    device_type,
    model,
    has_device_info,
    has_log,
    first_log_date,
    latest_stat_date,
    latest_status,
    total_alarm_count,
    error_days,
    warn_days,
    max_consecutive_error_days,
    equipment_risk_score,
    risk_rank,

    CASE
        WHEN has_log = FALSE THEN 'NO_LOG'
        WHEN latest_status = 'ERROR' THEN 'HIGH'
        WHEN max_consecutive_error_days >= 2 THEN 'HIGH'
        WHEN equipment_risk_score >= 35 THEN 'HIGH'
        WHEN error_days >= 1 THEN 'MEDIUM'
        WHEN latest_status = 'WARN' THEN 'MEDIUM'
        WHEN total_alarm_count >= 10 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS risk_level,

    CASE
        WHEN has_device_info = FALSE THEN 'UNKNOWN_DEVICE'
        WHEN has_log = FALSE THEN 'NO_LOG'
        ELSE 'OK'
    END AS data_quality_status

FROM df_final_merge
ORDER BY
    risk_rank,
    device_id;
'''
df_risk_level = duckdb.execute(query_risk_level).fetchdf()
df_risk_level

,device_id,site,runway,device_type,model,has_device_info,has_log,first_log_date,latest_stat_date,latest_status,total_alarm_count,error_days,warn_days,max_consecutive_error_days,equipment_risk_score,risk_rank,risk_level,data_quality_status
0,VIS_C,R35,RWY35,VIS,FS11,True,True,2026-07-01,2026-07-16,ERROR,80,9,0,3,155,1,HIGH,OK
1,VIS_A,R34,RWY34,VIS,FS11,True,True,2026-07-01,2026-07-07,ERROR,33,5,0,3,88,2,HIGH,OK
2,UNK_X,NaN,NaN,NaN,NaN,False,True,2026-07-01,2026-07-03,NORMAL,12,2,0,2,42,3,HIGH,UNKNOWN_DEVICE
3,BLS_E,R34,RWY34,BLS,BL100,True,True,2026-07-01,2026-07-06,NORMAL,7,1,1,1,22,4,MEDIUM,OK
4,RVR_B,R34,RWY34,RVR,LT31,True,True,2026-07-01,2026-07-07,NORMAL,5,0,1,0,5,5,LOW,OK
5,VIS_F,R36,RWY36,VIS,FS11,True,True,2026-07-03,2026-07-05,WARN,3,0,1,0,3,6,MEDIUM,OK
6,RVR_D,R35,RWY35,RVR,LT31,True,False,NaT,NaT,NaN,0,0,0,0,0,7,NO_LOG,NO_LOG
7,UNK_Y,NaN,NaN,NaN,NaN,False,True,2026-07-04,2026-07-04,NORMAL,0,0,0,0,0,7,LOW,UNKNOWN_DEVICE
